In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# %%capture --no-stderr
# !pip install python-dotenv openAI langchain_core langchain_openai

In [3]:
# 환경변수 설정

In [4]:
# 라이브러리 불러오기
from dotenv import load_dotenv
import os
from langchain_openai import OpenAI

In [5]:
# .env 파일에서 환경 변수 로드
load_dotenv("/content/.env")
# 환경 변수에서 API 키 가져오기
api_key = os.getenv("OPENAI_API_KEY")
# 오픈AI 대규모 언어 모델 초기화
llm = OpenAI()

In [6]:
llm.invoke("안녕")

'하세요. 이번에는 제가 만든 채팅 프로그램을 소개하려고 합니다.\n저는 최근에 코딩을 배우기 시작하면서 채팅 프로그램을 만들어보았습니다. 이 프로그램은 소켓 통신을 이용하여 서버와 클라이언트가 실시간으로 채팅을 주고받을 수 있도록 구현하였습니다.\n\n먼저 서버 프로그램을 실행하면 클라이언트가 접속하기를 기다립니다. 클라이언트가 접속하면 서버측에서는 해당 클라이언트의 접속을 확인하고, 클라이언트 측에서는 서버의 IP 주소를 입력하여 접속할 수 있도록 하였습니다.\n\n접속이 완료되면 두 사용자 간에 실시간으로 채팅'

In [7]:
# <오픈AI API를 사용하여 주제에 대한 간단한 설명 생성 파이프라인 구축>

# 라이브러리 불러오기
import openai
from typing import List

# 기본 오픈AI 클라이언트 사용
client = openai.OpenAI()

# "안녀하세요!" 메시지를 보내고 응답을 받음
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "안녕하세요!"}]
)
response.choices[0].message.content

'안녕하세요! 어떻게 도와드릴까요?'

In [8]:
# 요청에 사용할 프롬프트 템플릿 정의
prompt_template = "주제 {topic}에 대해 짧은 설명을 해주세요"

# 메시지를 보내고 모델의 응답을 받는 함수
def call_chat_model(messages: List[dict]) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )
    return response.choices[0].message.content

# 주어진 주제에 따라 설명을 요청하는 함수
def invoke_chain(topic: str) -> str:
    prompt_value = prompt_template.format(topic=topic)
    messages = [{"role": "user", "content": prompt_value}]
    return call_chat_model(messages)

# "더블딥" 주제로 설명 요청
invoke_chain("더블딥")

'더블딥(Double Dip)은 경제학에서 사용되는 용어로, 경기 침체가 끝난 후 잠시 회복세를 보이다가 다시 한번 경기 침체에 빠지는 현상을 지칭합니다. 이 상황은 보통 경제 지표가 일시적으로 개선되다가 다시 악화되면서 나타나며, 경제가 두 번 연속으로 침체에 들어간 듯한 모양새를 띱니다. 더블딥은 투자자와 정책 입안자에게 큰 위험 요소로 작용할 수 있습니다.'

In [9]:
# <랭체인을 사용하여 주제에 대한 간단한 설명 생성 파이프라인 구축>

# 라이브러리 불러오기
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 미스트랄AI 모델을 사용할 경우 주석 해제
# from langchain_mistralai.chat_models import ChatMistralAI

# 주어진 주제에 대해 짧은 설명을 요청하는 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_template(
    "주제 {topic}에 대해 짧은 설명을 해주세요"
)

# 출력 파서를 문자열로 설정
output_parser = StrOutputParser()
# 오픈AI의 gpt-4o 모델을 사용하여 채팅 모델 설정
model = ChatOpenAI(model="gpt-4o")
# 미스트랄AI 모델을 사용할 경우 주석 해제
# model = ChatMistralAI()

# 파이프라인 설정: 주제를 받아 프롬프트를 생성하고, 모델을 통해 응답을 생성한 후 문자열로 파싱
chain = (
    {"topic": RunnablePassthrough()}  # 입력 받은 주제를 그대로 통과시킴
    | prompt                         # 프롬프트 템플릿을 적용
    | model                          # 모델을 사용해 응답 생성
    | output_parser                  # 응답을 문자열로 파싱
)

# "더블딥" 주제로 설명 요청
chain.invoke("더블딥")

'"더블딥"이라는 용어는 주로 경제 분야에서 경기 침체를 설명할 때 사용됩니다. 이는 경제가 한 번 침체를 겪은 후 회복되는 듯하다가 다시 침체에 빠지는 현상을 의미합니다. 첫 번째 침체 이후의 회복은 일시적이며, 경제가 지속 가능한 성장을 이루지 못하고 두 번째 침체에 빠지는 것이 특징입니다. 이러한 더블딥 현상은 정책의 실패, 외부 충격, 혹은 구조적인 문제 등 다양한 요인에 의해 발생할 수 있습니다.'